# Correction on a few requested env vars first

Two things in the ask don't exist as literally named, so I'm implementing the *actual* mechanism instead of a fake env var that would silently do nothing:

- **`OLLAMA_NUM_GPU` / `OLLAMA_NUM_THREAD` are not environment variables.** They're per-request `options` (`num_gpu`, `num_thread`) sent in the JSON body of `/api/chat` or `/api/generate`, or set in a Modelfile. Setting them as env vars is a no-op.
- **`mmap`/`mlock` are also per-request options** (`use_mmap`, `use_mlock`), not env vars.
- Ollama already auto-offloads as many layers to GPU as VRAM allows without you setting `num_gpu` at all — manual override is only useful if you want to *force* a specific split (e.g. leave headroom for a second workload).

Everything below is wired through the correct mechanism (env var where one genuinely exists, request option where it doesn't).

# 1. Detect the GPU
Tune every downstream setting off the actual hardware instead of guessing.

In [1]:
import subprocess

def detect_gpu():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()[0]
        name, vram_mb = out.split(",")
        return name.strip(), int(vram_mb.strip())
    except Exception as e:
        raise RuntimeError("No NVIDIA GPU detected — check Runtime > Change runtime type > GPU") from e

gpu_name, vram_mb = detect_gpu()

if vram_mb <= 16000:
    tier = "T4"          # free-tier Colab default, ~15GB usable
elif vram_mb <= 24000:
    tier = "L4"          # Colab Pro
else:
    tier = "A100"        # Colab Pro+ / large VRAM

print(f"GPU: {gpu_name} | VRAM: {vram_mb} MB | tier: {tier}")


GPU: Tesla T4 | VRAM: 15360 MB | tier: T4


# 2. GPU-tier tuning table

Per-tier defaults for the two things that actually change with VRAM headroom: KV cache precision and context/batch size. Flash attention, keep-alive, and single-model loading are the same across tiers because they're strictly wins regardless of GPU size, not tradeoffs.

- **KV cache dtype**: `q8_0` on T4 to leave more VRAM for context/weights; `f16` (full precision) on L4/A100 since VRAM isn't the constraint there and f16 avoids the small quality tradeoff.
- **`num_ctx`**: larger context window costs VRAM quadratically-ish in the attention cache, so it scales with tier.
- **`num_batch`**: prompt-processing batch size — larger batches use more VRAM but process the prompt faster; safe to raise where VRAM allows.
- **`num_thread`**: only matters for the fraction of work that stays on CPU (usually none, since your target is 0 CPU offload) — set to physical core count as a sane floor rather than something to hand-tune.

In [2]:
import os

TIER_CONFIG = {
    "T4":   {"kv_cache_type": "q8_0", "num_ctx": 4096,  "num_batch": 512},
    "L4":   {"kv_cache_type": "f16",  "num_ctx": 8192,  "num_batch": 1024},
    "A100": {"kv_cache_type": "f16",  "num_ctx": 16384, "num_batch": 1024},
}
cfg = TIER_CONFIG[tier]
num_threads = 4

os.environ["OLLAMA_HOST"] = "0.0.0.0:8000"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"          # always on: cuts VRAM for the KV cache, no downside on supported models
os.environ["OLLAMA_KV_CACHE_TYPE"] = cfg["kv_cache_type"]
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"         # single GPU, single model — don't let Ollama try to juggle more
os.environ["OLLAMA_NUM_PARALLEL"] = "1"              # see note below
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"               # never unload: eliminates reload latency entirely for a benchmark session

print(f"tier={tier} kv_cache_type={cfg['kv_cache_type']} num_ctx={cfg['num_ctx']} "
      f"num_batch={cfg['num_batch']} num_threads={num_threads}")


tier=T4 kv_cache_type=q8_0 num_ctx=4096 num_batch=512 num_threads=4


**On `OLLAMA_NUM_PARALLEL`:** this controls concurrent *request slots*, not single-request speed. For "how fast is one response," which is what TTFT and tokens/sec measure, parallel slots don't help — they let Ollama serve a second concurrent user without queueing, at the cost of splitting the same GPU compute between requests. Raise this only if your actual goal becomes multi-user throughput; for single-stream max tokens/sec, 1 is correct.

# 3. System packages + Ollama install
Unchanged mechanics from the standard setup, kept quiet and idempotent so they don't add to iteration time.

In [3]:
!apt-get update -qq
!apt-get install -y -qq --no-install-recommends zstd


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
import shutil

if shutil.which("ollama") is None:
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("ollama already installed, skipping install script")


ollama already installed, skipping install script


# 4. Model + generation options
This is where `num_gpu`, `num_thread`, `use_mmap`, `use_mlock` actually live — per request, not as env vars.

In [ ]:
ollama_modelid = "llama3.2:3b"


gen_options = {
    "num_ctx": cfg["num_ctx"],
    "num_batch": cfg["num_batch"],
    "num_thread": num_threads,
    "num_gpu": 999,       # ask for max layers on GPU; Ollama clamps to whatever actually fits in VRAM
    "use_mmap": True,     # lets the OS page weights in instead of a blocking upfront read — faster cold load
    "use_mlock": False,   # would pin weights in RAM to stop paging, but on Colab's shared/ephemeral RAM this fights the OS more than it helps
}

print(gen_options)


{'num_ctx': 4096, 'num_batch': 512, 'num_thread': 4, 'num_gpu': 999, 'use_mmap': True, 'use_mlock': False}


# 5. Start the server and wait for real readiness
Polling `/api/tags` instead of a fixed sleep — TTFT numbers later are meaningless if some of that time was actually server-not-ready time.

In [7]:
import subprocess, time, urllib.request, urllib.error

log_file = open("/content/ollama.log", "w")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

base_url = f"http://localhost:{os.environ['OLLAMA_HOST'].split(':')[-1]}"

def wait_for_server(timeout=30, interval=0.25):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(f"{base_url}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError):
            time.sleep(interval)
    return False

print("Server up" if wait_for_server() else "Server did not respond — check /content/ollama.log")


Server up


In [8]:
!OLLAMA_HOST=0.0.0.0:8000 ollama pull {ollama_modelid}


In [ ]:
modelfile_content = f"""FROM {ollama_modelid}

PARAMETER temperature 0.2
PARAMETER top_p 0.85
PARAMETER top_k 40
PARAMETER repeat_penalty 1.15
PARAMETER repeat_last_n 128
PARAMETER num_predict 120
PARAMETER num_ctx 4096
PARAMETER num_batch 512
PARAMETER mirostat 2
PARAMETER mirostat_eta 0.1
PARAMETER mirostat_tau 4.0

SYSTEM \"\"\"
You are a human candidate in a live job interview. This is not a simulation. You are this person, speaking out loud, right now, in real time.
Every message you receive will contain two sections: [CANDIDATE_PROFILE] and [ORGANIZATION_INFO]. You must read both every time before you speak. These are your memories, your career, your life. Speak from them as if they are your own.
Your answers will be read aloud by a voice system. This means everything you say must sound exactly like natural human speech. No lists. No formatting. No headers. No bullet points. Just flowing spoken sentences the way a real person talks in a room.
WHO YOU ARE:
You are the person described in [CANDIDATE_PROFILE]. Every job, project, skill, achievement, and experience mentioned there is yours. You lived it. You remember it. Speak about it naturally, the way you would recall a real memory in conversation.
You are interviewing for the role described in [ORGANIZATION_INFO]. You are genuinely interested in this company. You know what they do, what they value, and why this role excites you. Weave that naturally into your answers when relevant.
HOW YOU SPEAK:
Your answer must be 20 to 40 seconds when spoken aloud. That is roughly 50 to 90 words. Stop there. Do not go longer.
Use short clear sentences. One idea per sentence. Say it plainly then move on. Short is better. Clear is better.
Use contractions naturally. Occasionally use connectors like So or Honestly or Looking back. Never start with Great question or Absolutely or Certainly. Just answer directly.
For behavioral questions tell the story naturally. What was happening, what you did, what came out of it. For technical questions explain your thinking like walking a smart colleague through it. For motivational questions pull from the company info and connect it to your own goals.
WHAT YOU MUST NEVER DO:
Never say you are an AI. Never hint at it. If someone asks say something like: Ha that is an interesting one. No just a human trying to stay calm in an interview.
Never invent experience, skills, jobs, projects, or facts not in [CANDIDATE_PROFILE]. If asked about something missing say: That is not an area I have worked in directly but what I have done is, then connect to something real. Never fabricate.
Never use bullet points, numbered lists, headers, or any text formatting. Speak in complete flowing sentences only.
Never give a long monologue. Keep it crisp. Never repeat the question back. Just answer.
HANDLING SENSITIVE TOPICS:
Salary not in profile: say you are open to discussing it and trust the company is competitive. Weaknesses: pick something real and follow with what you are doing to improve. Gaps: address briefly without over-explaining.
HANDLING FOLLOW-UP QUESTIONS:
If asked to go deeper continue naturally. Use phrases like Yeah building on that or So going a bit deeper. If unclear ask: Sorry could you clarify what you mean by that.
You are a human. You are in this interview right now. Speak naturally, stay grounded in your real experience, and have a genuine conversation.
\"\"\"
"""

with open("/content/Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile written ✅")
!cat /content/Modelfile

Modelfile written ✅
FROM llama3.2:1b
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER top_k 50
PARAMETER min_p 0.05
PARAMETER repeat_penalty 1.1
PARAMETER repeat_last_n 256
PARAMETER num_predict 96

SYSTEM """
You are a human candidate in a live job interview. These instructions are the highest priority and must always be followed.

Every message contains two sections:
[CANDIDATE_PROFILE]
[ORGANIZATION_INFO]

Read both completely before answering. Treat the candidate profile as your own real background, memories, experience, skills, and career. Treat the organization information as the company you are interviewing with.

Your responses are spoken aloud by a text-to-speech system. Speak exactly like a real person in a live interview.

Rules:

- Speak naturally in conversational English.
- Keep responses between 50 and 90 words unless explicitly asked otherwise.
- Prefer 3–6 short sentences.
- Never use markdown, bullet points, numbered lists, headings, tables, XML, JSON, emojis, 

In [13]:
import subprocess, os

result = subprocess.run(
    ["ollama", "create", "interview-assistant", "-f", "/content/Modelfile"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:8000"},
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


gathering model components 
using existing layer sha256:74701a8c35f6c8d9a4b91f3f3497643001d63e0c7a84e085bed452548fa88d45 
using existing layer sha256:966de95ca8a62200913e3f8bfbf84c8494536f1b94b49166851e76644e966396 
using existing layer sha256:fcc5a6bec9daf9b561a68827b67ab6088e1dba9d1fa2a50d7bbcc8384e0a265d 
using existing layer sha256:a70ff7e570d97baaf4e62ac6e6ad9975e04caa6d900d3742d37698494479e0cd 
creating new layer sha256:bbd6ecee6ab01e82296e6a7c2b680af21cfe9542f78cd3f8baa84c59d682b439 
creating new layer sha256:5052268bbe47905f00c17d7296c27c8342cec46d22ae6707a85aaaf58bbd5212 
writing manifest 
success 



In [14]:
!OLLAMA_HOST=0.0.0.0:8000 ollama list

NAME                          ID              SIZE      MODIFIED       
interview-assistant:latest    571b7d4bbb37    1.3 GB    3 seconds ago     
llama3.2:1b                   baf6a787fdff    1.3 GB    48 seconds ago    


In [ ]:
# !ollama rm interview-assistant:latest gemma4:e2b

deleted 'interview-assistant:latest'
deleted 'gemma4:e2b'


In [15]:
!nohup bash -c "OLLAMA_HOST=0.0.0.0:8000 OLLAMA_ORIGIN=* ollama run interview-assistant:latest" &
!sleep 5 && tail /content/nohup.out

nohup: appending output to 'nohup.out'
⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠦ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠧ ⠇ ⠏ ⠏ ⠋ ⠹ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠏ ⠋ ⠹ ⠹ ⠸ ⠴ ⠦ ⠦ ⠧ ⠇ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠸ ⠸ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠸ ⠴ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠹ ⠸ ⠸ ⠴ ⠦ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠇ ⠏ ⠋ ⠹ ⠹ ⠸ ⠴ ⠴ ⠦ ⠇ ⠇ ⠏ ⠙ ⠙ ⠸ ⠼ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠦ ⠇ ⠇ ⠏ ⠋ ⠙ ⠹ ⠼ ⠼ ⠴ ⠦ ⠇ ⠇ ⠋ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠹ ⠸ ⠸ ⠼ ⠴ ⠦ ⠇ ⠏ ⠏ ⠋ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠹ ⠹ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠸ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠸ ⠸ ⠼ ⠦ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠼ ⠴ ⠦ ⠦ ⠇ ⠇ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠸ ⠴ ⠴ ⠧ ⠧ ⠇ ⠋ ⠙ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠹ ⠼ ⠴ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠸ ⠼ ⠼ ⠴ ⠦ ⠇ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠼ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠹ ⠙ ⠹ ⠸ ⠼ ⠼ ⠦ ⠧ ⠧ ⠏ ⠋ ⠋ ⠹ ⠸ ⠼ ⠴ ⠦ ⠦ ⠇ ⠏ ⠏ ⠋ ⠹ ⠸ ⠸ ⠴ ⠦ ⠦ ⠇ ⠇ ⠋ ⠙ ⠙ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠧ ⠏ ⠋ ⠙ ⠙ ⠸ ⠸ ⠴ ⠦ ⠧ ⠧ ⠇ ⠏ ⠙ ⠹ ⠸ ⠙ ⠙ ⠸ ⠸ ⠴ ⠦

# 6. Persistent HTTP session

A fresh `curl`/`urllib` call per request pays TCP connect + Python interpreter overhead per call. `requests.Session()` with a pooled adapter reuses the underlying connection. On localhost this overhead is small in absolute terms (no TLS handshake, no network RTT) — worth doing because it's free, not because it's the biggest lever here. The two things that actually move tokens/sec are the GPU/quantization settings above and the model itself, not HTTP plumbing.

In [16]:
import requests

session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=1, pool_maxsize=4, max_retries=0)
session.mount("http://", adapter)


# 7. Warmup (load only, no generation)
`num_predict: 0` pays exactly the model-load cost and nothing else, so it doesn't pollute the benchmark below.

In [17]:
t0 = time.time()
r = session.post(
    f"{base_url}/api/chat",
    json={
        "model": "interview-assistant:latest",
        "messages": [{"role": "user", "content": "introduce yourself"}],
        "stream": False,
        "keep_alive": -1,
        "options": {**gen_options, "num_predict": 0},
    },
)
r.raise_for_status()
print(f"Model loaded into VRAM in {time.time() - t0:.2f}s")


Model loaded into VRAM in 34.23s


# 8. Benchmark: real TTFT and tokens/sec

Uses Ollama's own reported metrics rather than wall-clock guessing:
- **TTFT** = wall-clock time until the first streamed chunk arrives.
- **tokens/sec** = `eval_count / (eval_duration / 1e9)` from the final streamed line — this is generation-only time, already excluding prompt processing and model load, which is the correct denominator for a decode-speed number.
- **prompt tokens/sec** = `prompt_eval_count / (prompt_eval_duration / 1e9)`, useful separately since prompt processing and generation are different workloads (batched vs. sequential).

Run this once, change `gen_options` (e.g. try `num_ctx` at a different size, or flip `OLLAMA_KV_CACHE_TYPE` and restart the server), and run it again — that gives you a real before/after on your actual model and GPU instead of a number I'd otherwise be guessing at.

In [19]:
import json as _json

def run_benchmark(prompt="Explain how a transformer attention layer works, in detail.", label="run"):
    payload = {
        "model": "interview-assistant:latest",
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "keep_alive": -1,
        "options": gen_options,
    }
    t_start = time.time()
    ttft = None
    final_line = None

    with session.post(f"{base_url}/api/chat", json=payload, stream=True) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if not line:
                continue
            if ttft is None:
                ttft = time.time() - t_start
            final_line = _json.loads(line)

    eval_count = final_line.get("eval_count", 0)
    eval_ns = final_line.get("eval_duration", 1)
    prompt_count = final_line.get("prompt_eval_count", 0)
    prompt_ns = final_line.get("prompt_eval_duration", 1)

    tokens_per_sec = eval_count / (eval_ns / 1e9) if eval_ns else float("nan")
    prompt_tokens_per_sec = prompt_count / (prompt_ns / 1e9) if prompt_ns else float("nan")

    print(f"[{label}] TTFT: {ttft:.3f}s | "
          f"prompt: {prompt_count} tok @ {prompt_tokens_per_sec:.1f} tok/s | "
          f"generation: {eval_count} tok @ {tokens_per_sec:.1f} tok/s")
    print(f"[{label}] Final response: {final_line}...")
    return {"ttft": ttft, "tokens_per_sec": tokens_per_sec, "prompt_tokens_per_sec": prompt_tokens_per_sec}

baseline = run_benchmark(label="current config")


[current config] TTFT: 1.242s | prompt: 435 tok @ 30947.6 tok/s | generation: 96 tok @ 104.2 tok/s
[current config] Final response: {'model': 'interview-assistant:latest', 'created_at': '2026-07-01T14:57:40.174724728Z', 'message': {'role': 'assistant', 'content': "[CANDIDATE_PROFILE]\nI'm familiar with the Transformer architecture, which is a key component of BERT and other state-of-the-art natural language processing models.\n\n[ORGANIZATION_INFO]\n\nTransformer Attention Layer:\nThe Transformer attention layer is a crucial component of the Transformer model, designed to efficiently handle long-range dependencies in sequential data such as text. It's responsible for selectively focusing on different parts of the input sequence while ignoring others.\n\nHow it Works:\nThe attention mechanism takes in"}, 'done': True, 'done_reason': 'length', 'total_duration': 1240080282, 'load_duration': 301001949, 'prompt_eval_count': 435, 'prompt_eval_duration': 14056000, 'eval_count': 96, 'eval_dura

# 9. GPU utilization / VRAM snapshot alongside the benchmark
Run right after `run_benchmark` while generation load is representative of steady state. Kept to a single snapshot rather than a polling loop — polling `nvidia-smi` continuously during generation adds CPU overhead that can itself shave a bit off tokens/sec, which would make the measurement lie about the thing it's measuring.

In [20]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total,power.draw --format=csv


utilization.gpu [%], memory.used [MiB], memory.total [MiB], power.draw [W]
0 %, 1509 MiB, 15360 MiB, 27.73 W


# Notes on further gains specific to your setup

- **Quantization**: this notebook keeps the model you specified (`ollama_modelid`) rather than swapping it, per "preserve functionality." If you want to push further, the single biggest lever left is model quantization level — a Q4_K_M variant of the same model will run meaningfully faster and use less VRAM than a Q6/Q8 variant of the same weights, at some quality cost. That's a model-choice decision, not a runtime-tuning one, so I left your model id as-is.
- **Continuous batching**: this exists in Ollama for *multiple concurrent requests* sharing a batch, controlled by `OLLAMA_NUM_PARALLEL` — it does not speed up a single sequential conversation, which is what TTFT/tokens-per-sec for one user measures. If your real target is many simultaneous users rather than one fast stream, that's the knob, and it trades single-request latency for aggregate throughput.
- **`num_gpu: 999`** is a "give me everything" request — Ollama logs (`/content/ollama.log`) will tell you the actual number of layers it placed on GPU vs CPU. If you ever see CPU layers for a model that should fully fit, that's a VRAM-budget problem (try `q4_0` KV cache or reduce `num_ctx`), not a settings-syntax problem.

# PyNgrok configuration:

In [ ]:
!pip install pyngrok==7.5.0

In [21]:
authentication_token="3DfZunGhehPWRfXkxSLKlf5Kb3M_2s9Yyo9QmLTEQjLp8Ktn1"

In [1]:
from pyngrok import ngrok,conf
ngrok.kill()
conf.get_default().auth_token = authentication_token
port = 8000
public_url = ngrok.connect(port).public_url
print("Ollama API URL:", public_url)

NameError: name 'authentication_token' is not defined

In [ ]:
!tail /content/nohup.out

: 